# Hybrid Recommender Model Development

## Project
An Explainable Knowledge Graph-Driven Hybrid Recommender for Intelligent Task Allocation in Distributed Onboarding Operations

## Objective

The purpose of this notebook is to develop the recommendation component of the task allocation system.

The hybrid recommender combines:

- Content-Based Filtering
- Collaborative Filtering
- Knowledge Graph reasoning
- Agent availability
- Agent workload
- Historical agent performance

The aim is to recommend the most suitable onboarding agents for each onboarding task while also providing understandable reasons for the recommendation.

## Input Data

The datasets used in this notebook were prepared in the previous preprocessing notebook.

The main datasets are:

- Onboarding tasks
- Onboarding agents
- Historical task-agent assignments
- Training dataset
- Validation dataset
- Testing dataset

The historical assignment data was divided chronologically into:

- 70% training
- 15% validation
- 15% testing

## 1. Mounting Google Drive

Google Drive is mounted so that the processed datasets created during preprocessing can be accessed from this notebook.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## 2. Importing Required Libraries

The following Python libraries are used to build the recommendation system:

- Pandas for data handling and analysis
- NumPy for numerical operations
- Scikit-learn for content-based similarity calculations
- NetworkX for knowledge graph development
- Joblib for saving model components

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 3. Defining Dataset and Model Paths

The project folders are defined so that the processed datasets can be loaded and the trained recommender components can later be saved.

The processed folder contains the task, agent and historical assignment datasets.

The splits folder contains the training, validation and testing datasets.

In [ ]:
BASE_PATH = (
    "/content/drive/MyDrive/"
    "task-allocation-recommender"
)

PROCESSED_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "processed"
)

SPLITS_PATH = os.path.join(
    BASE_PATH,
    "data",
    "onboarding",
    "splits"
)

MODEL_PATH = os.path.join(
    BASE_PATH,
    "models"
)

os.makedirs(
    MODEL_PATH,
    exist_ok=True
)

print("Processed data:", PROCESSED_PATH)
print("Dataset splits:", SPLITS_PATH)
print("Models:", MODEL_PATH)

Processed data: /content/drive/MyDrive/task-allocation-recommender/data/onboarding/processed
Dataset splits: /content/drive/MyDrive/task-allocation-recommender/data/onboarding/splits
Models: /content/drive/MyDrive/task-allocation-recommender/models


## 4. Loading the Prepared Datasets

The processed datasets created in the preprocessing stage are loaded into the notebook.

The task dataset contains the onboarding requests.

The agent dataset contains the synthetic onboarding agent profiles.

The training, validation and testing datasets contain the historical task-agent assignment records used for model development and evaluation.

In [ ]:
tasks = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "onboarding_tasks.parquet"
    )
)

agents = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "onboarding_agents.parquet"
    )
)

assignments = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "historical_assignments.parquet"
    )
)

train_data = pd.read_parquet(
    os.path.join(
        SPLITS_PATH,
        "train.parquet"
    )
)

validation_data = pd.read_parquet(
    os.path.join(
        SPLITS_PATH,
        "validation.parquet"
    )
)

test_data = pd.read_parquet(
    os.path.join(
        SPLITS_PATH,
        "test.parquet"
    )
)

In [ ]:
print("Tasks:", tasks.shape)
print("Agents:", agents.shape)
print("Assignments:", assignments.shape)

print("\nTraining:", train_data.shape)
print("Validation:", validation_data.shape)
print("Testing:", test_data.shape)

Tasks: (100000, 16)
Agents: (500, 9)
Assignments: (100000, 9)

Training: (70000, 9)
Validation: (15000, 9)
Testing: (15000, 9)


### Dataset Loading Result

The prepared datasets were successfully loaded into the model development notebook.

The project contains:

- 100,000 onboarding tasks
- 500 synthetic onboarding agents
- 100,000 historical task-agent assignments
- 70,000 training records
- 15,000 validation records
- 15,000 testing records

These datasets will now be used to develop and evaluate the hybrid recommendation model.

## 5. Inspecting the Model Input Data

Before building the recommendation algorithms, the task and agent datasets are inspected to confirm that the required recommendation features are available.

In [ ]:
print("TASK COLUMNS")
print(tasks.columns.tolist())

print("\nAGENT COLUMNS")
print(agents.columns.tolist())

TASK COLUMNS
['task_id', 'task_date', 'source_state_nigeria', 'county', 'channel', 'task_type', 'task_complexity', 'id_verification_score', 'face_match_score', 'needs_id_review', 'needs_face_review', 'required_skills', 'priority', 'historical_completion_time', 'task_outcome', 'task_profile']

AGENT COLUMNS
['agent_id', 'agent_name', 'county', 'experience_years', 'skills', 'current_workload', 'availability_status', 'performance_score', 'historical_tasks']


In [ ]:
tasks.head()

,task_id,task_date,source_state_nigeria,county,channel,task_type,task_complexity,id_verification_score,face_match_score,needs_id_review,needs_face_review,required_skills,priority,historical_completion_time,task_outcome,task_profile
0,APP-0001794062,2023-01-01,Abuja (FCT),Nairobi,mobile_app,Enhanced KYC Onboarding,High,0.807320,0.938188,False,False,"Customer Onboarding, KYC Verification, Enhance...",High,5,True,Enhanced KYC Onboarding | mobile_app | Custome...
1,APP-0000314484,2023-01-01,Abia,Uasin Gishu,mobile_app,Enhanced KYC Onboarding,High,0.935026,0.786391,False,False,"Customer Onboarding, KYC Verification, Enhance...",High,4,True,Enhanced KYC Onboarding | mobile_app | Custome...
2,APP-0001793580,2023-01-01,Edo,Kajiado,mobile_app,Basic Customer Onboarding,Low,0.730425,0.743027,False,False,Customer Onboarding,Low,5,True,Basic Customer Onboarding | mobile_app | Custo...
3,APP-0000942860,2023-01-01,Benue,Kiambu,mobile_app,Standard KYC Onboarding,Medium,0.915678,0.672681,False,True,"Customer Onboarding, KYC Verification, Face Ve...",Medium,4,True,Standard KYC Onboarding | mobile_app | Custome...
4,APP-0000121719,2023-01-01,Lagos,Kiambu,web,Standard KYC Onboarding,Medium,0.580538,0.966328,True,False,"Customer Onboarding, KYC Verification, Identit...",Medium,12,True,Standard KYC Onboarding | web | Customer Onboa...


In [ ]:
agents.head()

,agent_id,agent_name,county,experience_years,skills,current_workload,availability_status,performance_score,historical_tasks
0,AGT-0001,Onboarding Agent 1,Nakuru,7,"Customer Onboarding, KYC Verification, Enhance...",0,Available,0.489,192
1,AGT-0002,Onboarding Agent 2,Machakos,6,"Customer Onboarding, KYC Verification, Enhance...",5,Busy,0.512,213
2,AGT-0003,Onboarding Agent 3,Nyeri,5,"Customer Onboarding, KYC Verification, Enhance...",3,Available,0.512,187
3,AGT-0004,Onboarding Agent 4,Nairobi,8,"Customer Onboarding, KYC Verification, Enhance...",3,Available,0.498,150
4,AGT-0005,Onboarding Agent 5,Kajiado,2,"Customer Onboarding, KYC Verification, Field O...",2,Available,0.576,138


## 6. Model Input Validation

The task and agent datasets are checked to ensure that the important fields required by the recommendation algorithms are available.

This prevents errors when building the content-based, collaborative and knowledge graph components.

In [ ]:
required_task_columns = [
    "task_id",
    "county",
    "task_type",
    "required_skills",
    "task_profile"
]

required_agent_columns = [
    "agent_id",
    "county",
    "skills",
    "availability_status",
    "current_workload",
    "performance_score"
]

print("Missing task columns:")

for column in required_task_columns:
    if column not in tasks.columns:
        print(column)

print("\nMissing agent columns:")

for column in required_agent_columns:
    if column not in agents.columns:
        print(column)

Missing task columns:

Missing agent columns:


In [ ]:
print(
    "Missing task values:",
    tasks[
        required_task_columns
    ].isnull().sum().sum()
)

print(
    "Missing agent values:",
    agents[
        required_agent_columns
    ].isnull().sum().sum()
)

Missing task values: 0
Missing agent values: 0


# Part A: Content-Based Filtering

## 7. Content-Based Recommendation

The content-based recommendation component compares the skills required by an onboarding task with the skills available for each onboarding agent.

The task required skills and agent skills are text-based attributes. TF-IDF is therefore used to convert the text into numerical vectors.

Cosine similarity is then used to measure how closely an agent's skills match the requirements of a task.

A higher cosine similarity score indicates a stronger skill match.

### Inspecting Task and Agent Skills

The required skills for tasks and the skills possessed by onboarding agents are inspected before applying TF-IDF.

In [ ]:
print("Sample task required skills:\n")

print(
    tasks[
        ["task_id", "task_type", "required_skills"]
    ].head()
)

print("\nSample agent skills:\n")

print(
    agents[
        ["agent_id", "skills"]
    ].head()
)

Sample task required skills:

          task_id                  task_type  \
0  APP-0001794062    Enhanced KYC Onboarding   
1  APP-0000314484    Enhanced KYC Onboarding   
2  APP-0001793580  Basic Customer Onboarding   
3  APP-0000942860    Standard KYC Onboarding   
4  APP-0000121719    Standard KYC Onboarding   

                                     required_skills  
0  Customer Onboarding, KYC Verification, Enhance...  
1  Customer Onboarding, KYC Verification, Enhance...  
2                                Customer Onboarding  
3  Customer Onboarding, KYC Verification, Face Ve...  
4  Customer Onboarding, KYC Verification, Identit...  

Sample agent skills:

   agent_id                                             skills
0  AGT-0001  Customer Onboarding, KYC Verification, Enhance...
1  AGT-0002  Customer Onboarding, KYC Verification, Enhance...
2  AGT-0003  Customer Onboarding, KYC Verification, Enhance...
3  AGT-0004  Customer Onboarding, KYC Verification, Enhance...
4  AGT-0005  

### TF-IDF Feature Representation

TF-IDF converts the text-based task requirements and agent skills into numerical vectors.

This makes it possible for the system to mathematically compare a task with an agent.

The TF-IDF model is fitted using both task required skills and agent skills so that they share the same feature space.

In [ ]:
skill_corpus = pd.concat(
    [
        tasks["required_skills"]
        .fillna("")
        .astype(str),

        agents["skills"]
        .fillna("")
        .astype(str)
    ],
    ignore_index=True
)

print(
    "Number of text records:",
    len(skill_corpus)
)

Number of text records: 100500


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2)
)

In [ ]:
tfidf_vectorizer.fit(
    skill_corpus
)

TfidfVectorizer(ngram_range=(1, 2))

In [ ]:
agent_skill_vectors = (
    tfidf_vectorizer.transform(
        agents["skills"]
        .fillna("")
        .astype(str)
    )
)

In [ ]:
print(
    "TF-IDF vocabulary size:",
    len(
        tfidf_vectorizer.vocabulary_
    )
)

print(
    "Agent skill matrix shape:",
    agent_skill_vectors.shape
)

TF-IDF vocabulary size: 28
Agent skill matrix shape: (500, 28)


### TF-IDF Result

The TF-IDF model successfully converted the onboarding agent skills into numerical vectors.

Each agent is now represented numerically according to the skills in their profile.

These vectors will be compared with the required skills of each task using cosine similarity.

## 8. Testing Content-Based Similarity

One onboarding task is selected from the testing dataset.

Using a task from the testing data helps demonstrate how the recommender would handle a task that was not used during model training.

In [ ]:
tasks["task_id"] = (
    tasks["task_id"]
    .astype(str)
)

test_data["task_id"] = (
    test_data["task_id"]
    .astype(str)
)

agents["agent_id"] = (
    agents["agent_id"]
    .astype(str)
)

In [ ]:
sample_task_id = str(
    test_data.iloc[0]["task_id"]
)

sample_task = tasks[
    tasks["task_id"] == sample_task_id
].iloc[0]

print(
    "Task ID:",
    sample_task["task_id"]
)

print(
    "Task Type:",
    sample_task["task_type"]
)

print(
    "County:",
    sample_task["county"]
)

print(
    "Required Skills:",
    sample_task["required_skills"]
)

Task ID: APP-0000209575
Task Type: Standard KYC Onboarding
County: Kakamega
Required Skills: Customer Onboarding, KYC Verification


### Task Vector Creation

The selected task's required skills are converted into the same TF-IDF feature space used for the agent skill profiles.

In [ ]:
task_vector = (
    tfidf_vectorizer.transform(
        [
            str(
                sample_task[
                    "required_skills"
                ]
            )
        ]
    )
)

In [ ]:
print(
    "Task vector shape:",
    task_vector.shape
)

Task vector shape: (1, 28)


### Cosine Similarity

Cosine similarity is used to compare the selected task vector with every onboarding agent vector.

The score ranges approximately from 0 to 1.

A score closer to 1 means that the agent's skills closely match the skills required by the onboarding task.

In [ ]:
content_scores = (
    cosine_similarity(
        task_vector,
        agent_skill_vectors
    )
    .flatten()
)

In [ ]:
print(
    "Number of agents scored:",
    len(content_scores)
)

print(
    "Highest similarity score:",
    content_scores.max()
)

print(
    "Lowest similarity score:",
    content_scores.min()
)

Number of agents scored: 500
Highest similarity score: 1.0000000000000002
Lowest similarity score: 0.15287381495054916


In [ ]:
content_results = agents.copy()

content_results[
    "content_score"
] = content_scores

In [ ]:
content_results = (
    content_results
    .sort_values(
        "content_score",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
top_5_content = content_results[
    [
        "agent_id",
        "county",
        "skills",
        "availability_status",
        "current_workload",
        "performance_score",
        "content_score"
    ]
].head(5)

top_5_content

,agent_id,county,skills,availability_status,current_workload,performance_score,content_score
0,AGT-0040,Nairobi,"Customer Onboarding, KYC Verification",Available,0,0.488,1.0
1,AGT-0305,Nyeri,"Customer Onboarding, KYC Verification",Available,2,0.547,1.0
2,AGT-0344,Mombasa,"Customer Onboarding, KYC Verification",Available,2,0.523,1.0
3,AGT-0351,Nairobi,"Customer Onboarding, KYC Verification",Available,2,0.569,1.0
4,AGT-0089,Kakamega,"Customer Onboarding, KYC Verification",Available,1,0.525,1.0


In [ ]:
top_5_content = (
    top_5_content.copy()
)

top_5_content.insert(
    0,
    "rank",
    range(
        1,
        len(top_5_content) + 1
    )
)

top_5_content[
    "match_percentage"
] = (
    top_5_content[
        "content_score"
    ]
    * 100
).round(2)

top_5_content

,rank,agent_id,county,skills,availability_status,current_workload,performance_score,content_score,match_percentage
0,1,AGT-0040,Nairobi,"Customer Onboarding, KYC Verification",Available,0,0.488,1.0,100.0
1,2,AGT-0305,Nyeri,"Customer Onboarding, KYC Verification",Available,2,0.547,1.0,100.0
2,3,AGT-0344,Mombasa,"Customer Onboarding, KYC Verification",Available,2,0.523,1.0,100.0
3,4,AGT-0351,Nairobi,"Customer Onboarding, KYC Verification",Available,2,0.569,1.0,100.0
4,5,AGT-0089,Kakamega,"Customer Onboarding, KYC Verification",Available,1,0.525,1.0,100.0


### Content-Based Recommendation Result

The content-based model successfully ranked onboarding agents according to how closely their skills matched the requirements of the selected onboarding task.

The agents with the highest cosine similarity scores were placed at the top of the recommendation list.

At this stage, the recommendation only focuses mainly on skill similarity. Other factors such as historical task assignments, location, workload, availability and knowledge graph relationships will be included later in the hybrid model.

# Part B: Collaborative Filtering

## 9. Collaborative Filtering

The collaborative filtering component uses historical task-agent assignments to learn patterns from previous allocations.

The system examines which types of onboarding tasks have previously been handled by each agent.

An interaction matrix is created where:

- Rows represent onboarding agents
- Columns represent task profiles
- Values represent the number of historical assignments

Cosine similarity is then used to identify agents with similar historical task-handling patterns.

This allows the recommender to consider previous experience in addition to the direct skill matching used by the content-based model.

### Inspecting Historical Assignments

The training dataset is inspected to confirm that it contains the agent IDs and task profiles required for collaborative filtering.

In [ ]:
train_data[
    [
        "task_id",
        "agent_id",
        "task_profile"
    ]
].head(10)

,task_id,agent_id,task_profile
0,APP-0000006185,AGT-0015,Standard KYC Onboarding | mobile_app | Custome...
1,APP-0000009682,AGT-0196,Basic Customer Onboarding | mobile_app | Custo...
2,APP-0000027949,AGT-0321,Standard KYC Onboarding | web | Customer Onboa...
3,APP-0000037037,AGT-0049,Standard KYC Onboarding | web | Customer Onboa...
4,APP-0000045900,AGT-0436,Standard KYC Onboarding | web | Customer Onboa...
5,APP-0000048556,AGT-0442,Enhanced KYC Onboarding | mobile_app | Custome...
6,APP-0000094402,AGT-0352,Standard KYC Onboarding | web | Customer Onboa...
7,APP-0000121719,AGT-0283,Standard KYC Onboarding | web | Customer Onboa...
8,APP-0000142756,AGT-0270,Enhanced KYC Onboarding | mobile_app | Custome...
9,APP-0000143560,AGT-0227,Basic Customer Onboarding | mobile_app | Custo...


In [ ]:
print(
    "Training assignments:",
    len(train_data)
)

print(
    "Unique agents:",
    train_data["agent_id"].nunique()
)

print(
    "Unique task profiles:",
    train_data["task_profile"].nunique()
)

Training assignments: 70000
Unique agents: 500
Unique task profiles: 36


## 10. Creating the Agent-Task Profile Interaction Matrix

An interaction matrix is created from the training dataset.

Each row represents an onboarding agent and each column represents a task profile.

The values show how many times an agent previously handled that type of task.

In [ ]:
interaction_matrix = pd.crosstab(
    train_data["agent_id"],
    train_data["task_profile"]
)

In [ ]:
interaction_matrix = (
    interaction_matrix
    .reindex(
        agents["agent_id"],
        fill_value=0
    )
)

In [ ]:
print(
    "Interaction matrix shape:",
    interaction_matrix.shape
)

interaction_matrix.head()

Interaction matrix shape: (500, 36)


task_profile,"Basic Customer Onboarding | agent | Customer Onboarding, Face Verification, Field Onboarding","Basic Customer Onboarding | agent | Customer Onboarding, Field Onboarding","Basic Customer Onboarding | agent | Customer Onboarding, Identity Verification, Face Verification, Field Onboarding","Basic Customer Onboarding | agent | Customer Onboarding, Identity Verification, Field Onboarding",Basic Customer Onboarding | mobile_app | Customer Onboarding,"Basic Customer Onboarding | mobile_app | Customer Onboarding, Face Verification","Basic Customer Onboarding | mobile_app | Customer Onboarding, Identity Verification","Basic Customer Onboarding | mobile_app | Customer Onboarding, Identity Verification, Face Verification",Basic Customer Onboarding | web | Customer Onboarding,"Basic Customer Onboarding | web | Customer Onboarding, Face Verification",...,"Standard KYC Onboarding | agent | Customer Onboarding, KYC Verification, Identity Verification, Face Verification, Field Onboarding","Standard KYC Onboarding | agent | Customer Onboarding, KYC Verification, Identity Verification, Field Onboarding","Standard KYC Onboarding | mobile_app | Customer Onboarding, KYC Verification","Standard KYC Onboarding | mobile_app | Customer Onboarding, KYC Verification, Face Verification","Standard KYC Onboarding | mobile_app | Customer Onboarding, KYC Verification, Identity Verification","Standard KYC Onboarding | mobile_app | Customer Onboarding, KYC Verification, Identity Verification, Face Verification","Standard KYC Onboarding | web | Customer Onboarding, KYC Verification","Standard KYC Onboarding | web | Customer Onboarding, KYC Verification, Face Verification","Standard KYC Onboarding | web | Customer Onboarding, KYC Verification, Identity Verification","Standard KYC Onboarding | web | Customer Onboarding, KYC Verification, Identity Verification, Face Verification"
agent_id,,,,,,,,,,,,,,,,,,,,,
AGT-0001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
AGT-0002,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AGT-0003,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AGT-0004,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
AGT-0005,0,4,0,0,1,0,0,0,0,0,...,0,0,75,0,0,0,27,0,0,0


### Interaction Matrix Result

The interaction matrix contains 500 onboarding agents and 36 task profiles.

Each row represents an onboarding agent, while each column represents a specific task profile.

The values show how many times each agent handled that task profile in the historical training data.

A value of 0 means that the agent did not handle that task profile, while higher values show more historical experience with that type of task.

## 11. Calculating Agent Similarity

Cosine similarity is applied to the interaction matrix to measure similarity between onboarding agents.

Agents with similar historical task assignment patterns receive higher similarity scores.

In [ ]:
agent_similarity = cosine_similarity(
    interaction_matrix
)

In [ ]:
print(
    "Agent similarity matrix:",
    agent_similarity.shape
)

Agent similarity matrix: (500, 500)


In [ ]:
np.fill_diagonal(
    agent_similarity,
    0
)

In [ ]:
print(
    "Highest similarity after removing self:",
    agent_similarity.max()
)

Highest similarity after removing self: 1.0000000000000002


## 12. Generating Collaborative Recommendation Scores

Collaborative scores are generated by combining the historical task profiles of similar agents.

This produces an estimated suitability score for every agent and task profile.

The scores are then normalized to a range between 0 and 1 for easier comparison with the other recommendation components.

In [ ]:
interaction_values = (
    interaction_matrix
    .to_numpy(
        dtype=float
    )
)

In [ ]:
similarity_sum = (
    agent_similarity
    .sum(
        axis=1,
        keepdims=True
    )
)

similarity_sum[
    similarity_sum == 0
] = 1

In [ ]:
collaborative_raw = (
    agent_similarity
    @ interaction_values
) / similarity_sum

In [ ]:
print(
    "Collaborative matrix shape:",
    collaborative_raw.shape
)

Collaborative matrix shape: (500, 36)


In [ ]:
profile_maximums = (
    collaborative_raw
    .max(
        axis=0,
        keepdims=True
    )
)

profile_maximums[
    profile_maximums == 0
] = 1

collaborative_scores = (
    collaborative_raw
    / profile_maximums
)

In [ ]:
print(
    "Minimum collaborative score:",
    collaborative_scores.min()
)

print(
    "Maximum collaborative score:",
    collaborative_scores.max()
)

Minimum collaborative score: 0.0
Maximum collaborative score: 1.0


In [ ]:
profile_to_column = {
    profile: index
    for index, profile
    in enumerate(
        interaction_matrix.columns
    )
}

In [ ]:
print(
    "Number of task profiles:",
    len(profile_to_column)
)

Number of task profiles: 36


## 13. Collaborative Recommendation for the Selected Task

The same testing task used for content-based filtering is used again.

Its task profile is matched against the historical interaction matrix to generate collaborative recommendation scores.

In [ ]:
print(
    "Task ID:",
    sample_task["task_id"]
)

print(
    "Task Profile:",
    sample_task["task_profile"]
)

Task ID: APP-0000209575
Task Profile: Standard KYC Onboarding | mobile_app | Customer Onboarding, KYC Verification


In [ ]:
sample_profile = (
    sample_task[
        "task_profile"
    ]
)

In [ ]:
if sample_profile in profile_to_column:

    profile_column = (
        profile_to_column[
            sample_profile
        ]
    )

    sample_cf_scores = (
        collaborative_scores[
            :,
            profile_column
        ]
    )

else:

    sample_cf_scores = np.zeros(
        len(agents)
    )

    print(
        "Cold-start task profile detected"
    )

In [ ]:
cf_results = agents.copy()

cf_results[
    "collaborative_score"
] = sample_cf_scores

In [ ]:
cf_results = (
    cf_results
    .sort_values(
        "collaborative_score",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
top_5_cf = cf_results[
    [
        "agent_id",
        "county",
        "skills",
        "availability_status",
        "current_workload",
        "performance_score",
        "collaborative_score"
    ]
].head(5)

In [ ]:
top_5_cf = (
    top_5_cf.copy()
)

top_5_cf.insert(
    0,
    "rank",
    range(
        1,
        len(top_5_cf) + 1
    )
)

In [ ]:
top_5_cf[
    "collaborative_percentage"
] = (
    top_5_cf[
        "collaborative_score"
    ]
    * 100
).round(2)

top_5_cf

,rank,agent_id,county,skills,availability_status,current_workload,performance_score,collaborative_score,collaborative_percentage
0,1,AGT-0155,Kakamega,"Customer Onboarding, KYC Verification",Available,0,0.562,1.000000,100.00
1,2,AGT-0089,Kakamega,"Customer Onboarding, KYC Verification",Available,1,0.525,0.999912,99.99
2,3,AGT-0341,Kakamega,"Customer Onboarding, KYC Verification",Available,1,0.480,0.999912,99.99
3,4,AGT-0237,Machakos,"Customer Onboarding, KYC Verification",Available,2,0.480,0.998691,99.87
4,5,AGT-0177,Machakos,"Customer Onboarding, KYC Verification",Busy,4,0.534,0.998691,99.87


### Collaborative Filtering Result

The collaborative filtering component ranked agents according to patterns learned from historical task-agent assignments.

Unlike the content-based model, which mainly compares skills, collaborative filtering considers which agents have previously handled similar task profiles.

This allows historical task allocation patterns to contribute to the recommendation process.

The collaborative component will later be combined with content-based filtering, knowledge graph reasoning and operational factors to create the final hybrid recommender.

# Part C: Knowledge Graph

## 14. Knowledge Graph Development

A knowledge graph is created to represent relationships between the main entities in the task allocation system.

The graph represents relationships such as:

- Onboarding Agent HAS_SKILL Skill
- Onboarding Agent LOCATED_IN County
- Onboarding Agent HANDLED Task Type
- Task REQUIRES Skill
- Task LOCATED_IN County

The knowledge graph helps the system reason about the relationship between onboarding tasks and agents.

It also supports explainability because the system can show why a particular agent is suitable for a task.

### Preparing Skill Relationships

The skills stored in the task and agent datasets are separated into individual values so that each skill can be represented as a separate node in the knowledge graph.

In [ ]:
def split_skills(skill_text):

    if pd.isna(skill_text):
        return []

    return [
        skill.strip()
        for skill in str(skill_text).split(",")
        if skill.strip()
    ]

In [ ]:
print(
    split_skills(
        agents.iloc[0]["skills"]
    )
)

['Customer Onboarding', 'KYC Verification', 'Enhanced KYC Review', 'Identity Verification', 'Face Verification']


In [ ]:
all_skills = set()

for skills in agents["skills"]:

    all_skills.update(
        split_skills(skills)
    )

for skills in tasks["required_skills"]:

    all_skills.update(
        split_skills(skills)
    )

all_skills = sorted(all_skills)

In [ ]:
print(
    "Number of unique skills:",
    len(all_skills)
)

print("\nSkills:")

for skill in all_skills:
    print("-", skill)

Number of unique skills: 6

Skills:
- Customer Onboarding
- Enhanced KYC Review
- Face Verification
- Field Onboarding
- Identity Verification
- KYC Verification


### Skill Nodes

The required and available skills were extracted from the task and agent datasets.

Each unique skill will be represented as a node in the knowledge graph.

These skill nodes create connections between tasks that require a skill and agents that possess the same skill.

## 15. Creating the Knowledge Graph

NetworkX is used to create the prototype knowledge graph.

Nodes represent entities such as:

- onboarding agents
- skills
- counties
- task types

Edges represent the relationships between these entities.

In [ ]:
G = nx.Graph()

In [ ]:
for _, agent in agents.iterrows():

    agent_node = (
        "AGENT_"
        + str(agent["agent_id"])
    )

    G.add_node(
        agent_node,
        node_type="Agent"
    )

In [ ]:
print(
    "Nodes after adding agents:",
    G.number_of_nodes()
)

Nodes after adding agents: 500


## 16. Adding Agent-Skill Relationships

Each onboarding agent is connected to the skills contained in their profile.

The relationship `HAS_SKILL` represents the skills that an agent possesses.

These relationships allow the knowledge graph to identify agents whose skills match the requirements of an onboarding task.

In [ ]:
for _, agent in agents.iterrows():

    agent_node = (
        "AGENT_"
        + str(agent["agent_id"])
    )

    agent_skills = split_skills(
        agent["skills"]
    )

    for skill in agent_skills:

        skill_node = (
            "SKILL_"
            + skill
        )

        G.add_node(
            skill_node,
            node_type="Skill",
            label=skill
        )

        G.add_edge(
            agent_node,
            skill_node,
            relationship="HAS_SKILL"
        )

In [ ]:
print(
    "Total nodes:",
    G.number_of_nodes()
)

print(
    "Total edges:",
    G.number_of_edges()
)

Total nodes: 506
Total edges: 1720


## 17. Adding Agent-Location Relationships

Each onboarding agent is connected to their assigned Kenyan county.

The `LOCATED_IN` relationship allows the recommendation system to identify agents who are located in the same area as an onboarding task.

In [ ]:
for _, agent in agents.iterrows():

    agent_node = (
        "AGENT_"
        + str(agent["agent_id"])
    )

    county_node = (
        "COUNTY_"
        + str(agent["county"])
    )

    G.add_node(
        county_node,
        node_type="County",
        label=agent["county"]
    )

    G.add_edge(
        agent_node,
        county_node,
        relationship="LOCATED_IN"
    )

In [ ]:
print(
    "Total nodes:",
    G.number_of_nodes()
)

print(
    "Total edges:",
    G.number_of_edges()
)

Total nodes: 516
Total edges: 2220


## 18. Preparing Historical Task-Type Relationships

The training assignment records are combined with the task dataset to identify the types of onboarding tasks previously handled by each agent.

Only the training dataset is used so that information from validation and testing data does not leak into model development.

In [ ]:
train_with_task_type = (
    train_data.merge(
        tasks[
            [
                "task_id",
                "task_type"
            ]
        ],
        on="task_id",
        how="left"
    )
)

In [ ]:
train_with_task_type[
    [
        "task_id",
        "agent_id",
        "task_type"
    ]
].head()

,task_id,agent_id,task_type
0,APP-0000006185,AGT-0015,Standard KYC Onboarding
1,APP-0000009682,AGT-0196,Basic Customer Onboarding
2,APP-0000027949,AGT-0321,Standard KYC Onboarding
3,APP-0000037037,AGT-0049,Standard KYC Onboarding
4,APP-0000045900,AGT-0436,Standard KYC Onboarding


In [ ]:
print(
    "Missing task types:",
    train_with_task_type[
        "task_type"
    ].isnull().sum()
)

Missing task types: 0


## 19. Adding Historical Task-Type Relationships

Historical task allocation records are used to connect agents to the types of onboarding tasks they previously handled.

The `HANDLED_TYPE` relationship represents an agent's previous experience with a particular onboarding task type.

The number of historical assignments is stored as an edge attribute called `count`.

In [ ]:
historical_task_types = (
    train_with_task_type
    .groupby(
        [
            "agent_id",
            "task_type"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

In [ ]:
historical_task_types.head(10)

,agent_id,task_type,count
0,AGT-0001,Enhanced KYC Onboarding,191
1,AGT-0001,Standard KYC Onboarding,1
2,AGT-0002,Enhanced KYC Onboarding,213
3,AGT-0003,Enhanced KYC Onboarding,187
4,AGT-0004,Enhanced KYC Onboarding,149
5,AGT-0004,Standard KYC Onboarding,1
6,AGT-0005,Basic Customer Onboarding,5
7,AGT-0005,Standard KYC Onboarding,133
8,AGT-0006,Basic Customer Onboarding,107
9,AGT-0007,Basic Customer Onboarding,1


In [ ]:
for _, row in historical_task_types.iterrows():

    agent_node = (
        "AGENT_"
        + str(row["agent_id"])
    )

    task_type_node = (
        "TASK_TYPE_"
        + str(row["task_type"])
    )

    G.add_node(
        task_type_node,
        node_type="TaskType",
        label=row["task_type"]
    )

    G.add_edge(
        agent_node,
        task_type_node,
        relationship="HANDLED_TYPE",
        count=int(
            row["count"]
        )
    )

In [ ]:
print("KNOWLEDGE GRAPH SUMMARY")

print(
    "Total nodes:",
    G.number_of_nodes()
)

print(
    "Total edges:",
    G.number_of_edges()
)

KNOWLEDGE GRAPH SUMMARY
Total nodes: 519
Total edges: 2889


### Knowledge Graph Relationship Result

The knowledge graph now contains onboarding agents and their relationships with:

- Skills
- Kenyan counties
- Previously handled onboarding task types

These relationships provide additional information that can later be used when ranking agents and generating explanations for recommendations.

## 20. Adding the Current Task to the Knowledge Graph

When a task requires an agent recommendation, the task is represented in the knowledge graph together with its important relationships.

The task is connected to:

- its required skills using `REQUIRES`
- its county using `LOCATED_IN`
- its task type using `HAS_TYPE`

For the prototype, tasks can be added dynamically when recommendations are required instead of permanently adding all 100,000 historical tasks to the graph.

In [ ]:
task_node = (
    "TASK_"
    + str(sample_task["task_id"])
)

G.add_node(
    task_node,
    node_type="Task",
    label=sample_task["task_id"]
)

In [ ]:
print(
    "Task node added:",
    task_node
)

print(
    "Total nodes:",
    G.number_of_nodes()
)

Task node added: TASK_APP-0000209575
Total nodes: 520


### Task-Skill Relationships

The selected onboarding task is connected to every skill required to complete it.

The `REQUIRES` relationship allows the knowledge graph to identify agents who possess the same skills.

In [ ]:
required_task_skills = split_skills(
    sample_task["required_skills"]
)

for skill in required_task_skills:

    skill_node = (
        "SKILL_"
        + skill
    )

    G.add_node(
        skill_node,
        node_type="Skill",
        label=skill
    )

    G.add_edge(
        task_node,
        skill_node,
        relationship="REQUIRES"
    )

In [ ]:
print(
    "Task required skills:",
    required_task_skills
)

print(
    "Total edges:",
    G.number_of_edges()
)

Task required skills: ['Customer Onboarding', 'KYC Verification']
Total edges: 2891


### Task-Location Relationship

The onboarding task is connected to its Kenyan county using the `LOCATED_IN` relationship.

This allows the recommendation model to identify agents located in the same county as the task.

In [ ]:
task_county_node = (
    "COUNTY_"
    + str(sample_task["county"])
)

G.add_node(
    task_county_node,
    node_type="County",
    label=sample_task["county"]
)

G.add_edge(
    task_node,
    task_county_node,
    relationship="LOCATED_IN"
)

### Task-Type Relationship

The selected task is connected to its onboarding task type.

This allows historical agent experience with similar task types to contribute to the knowledge graph recommendation score.

In [ ]:
task_type_node = (
    "TASK_TYPE_"
    + str(sample_task["task_type"])
)

G.add_node(
    task_type_node,
    node_type="TaskType",
    label=sample_task["task_type"]
)

G.add_edge(
    task_node,
    task_type_node,
    relationship="HAS_TYPE"
)

In [ ]:
print("UPDATED KNOWLEDGE GRAPH")

print(
    "Total nodes:",
    G.number_of_nodes()
)

print(
    "Total edges:",
    G.number_of_edges()
)

print(
    "\nTask relationships:",
    list(
        G.edges(
            task_node,
            data=True
        )
    )
)

UPDATED KNOWLEDGE GRAPH
Total nodes: 520
Total edges: 2893

Task relationships: [('TASK_APP-0000209575', 'SKILL_Customer Onboarding', {'relationship': 'REQUIRES'}), ('TASK_APP-0000209575', 'SKILL_KYC Verification', {'relationship': 'REQUIRES'}), ('TASK_APP-0000209575', 'COUNTY_Kakamega', {'relationship': 'LOCATED_IN'}), ('TASK_APP-0000209575', 'TASK_TYPE_Standard KYC Onboarding', {'relationship': 'HAS_TYPE'})]


### Current Task Knowledge Graph Result

The selected onboarding task was successfully added to the knowledge graph.

The task is now connected to its required skills, location and task type.

These relationships can be compared with the relationships already stored for onboarding agents. This allows the system to identify agents who share relevant skills, location and historical task experience with the current task.

## 21. Knowledge Graph Recommendation Scoring

The knowledge graph recommendation score considers three forms of relationship between a task and an agent:

1. Required skill coverage
2. Location match
3. Historical experience with the same task type

These factors are combined to calculate a knowledge graph score for each onboarding agent.

In [ ]:
def kg_skill_match(
    task_skills,
    agent_skills
):

    required = set(
        split_skills(task_skills)
    )

    available = set(
        split_skills(agent_skills)
    )

    if len(required) == 0:
        return 0

    matched = required.intersection(
        available
    )

    return (
        len(matched)
        / len(required)
    )

In [ ]:
print(
    kg_skill_match(
        sample_task["required_skills"],
        agents.iloc[0]["skills"]
    )
)

1.0


## 22. Calculating the Knowledge Graph Score

The knowledge graph recommendation score combines three relationships between an onboarding task and an agent:

- Skill relationship: whether the agent has the skills required by the task
- Location relationship: whether the agent and task are in the same county
- Historical task-type relationship: whether the agent has previously handled similar onboarding tasks

For the prototype, the initial knowledge graph score is calculated as:

- 60% Skill Match
- 20% Location Match
- 20% Historical Task-Type Experience

These weights provide an initial representation of the importance of the relationships and can later be refined during model evaluation.

In [ ]:
kg_results = agents.copy()

In [ ]:
kg_results["skill_score"] = (
    kg_results["skills"]
    .apply(
        lambda agent_skills:
        kg_skill_match(
            sample_task["required_skills"],
            agent_skills
        )
    )
)

In [ ]:
sample_task_type = (
    sample_task["task_type"]
)

task_type_history = (
    historical_task_types[
        historical_task_types[
            "task_type"
        ] == sample_task_type
    ]
    .set_index("agent_id")["count"]
)

In [ ]:
kg_results[
    "historical_type_count"
] = (
    kg_results["agent_id"]
    .map(task_type_history)
    .fillna(0)
)

In [ ]:
max_history = (
    kg_results[
        "historical_type_count"
    ].max()
)

if max_history == 0:
    max_history = 1

kg_results[
    "task_type_score"
] = (
    kg_results[
        "historical_type_count"
    ]
    / max_history
)

In [ ]:
kg_results["location_score"] = (
    kg_results["county"]
    .astype(str)
    .eq(
        str(sample_task["county"])
    )
    .astype(float)
)

print(
    kg_results[
        [
            "agent_id",
            "county",
            "location_score"
        ]
    ].head()
)

   agent_id    county  location_score
0  AGT-0001    Nakuru             0.0
1  AGT-0002  Machakos             0.0
2  AGT-0003     Nyeri             0.0
3  AGT-0004   Nairobi             0.0
4  AGT-0005   Kajiado             0.0


In [ ]:
print(
    kg_results[
        [
            "skill_score",
            "location_score",
            "task_type_score"
        ]
    ].head()
)

   skill_score  location_score  task_type_score
0          1.0             0.0         0.005495
1          1.0             0.0         0.000000
2          1.0             0.0         0.000000
3          1.0             0.0         0.005495
4          1.0             0.0         0.730769


In [ ]:
kg_results["kg_score"] = (
    0.60
    * kg_results["skill_score"]

    + 0.20
    * kg_results["location_score"]

    + 0.20
    * kg_results["task_type_score"]
)

In [ ]:
kg_results[
    [
        "agent_id",
        "skill_score",
        "location_score",
        "task_type_score",
        "kg_score"
    ]
].head()

,agent_id,skill_score,location_score,task_type_score,kg_score
0,AGT-0001,1.0,0.0,0.005495,0.601099
1,AGT-0002,1.0,0.0,0.000000,0.600000
2,AGT-0003,1.0,0.0,0.000000,0.600000
3,AGT-0004,1.0,0.0,0.005495,0.601099
4,AGT-0005,1.0,0.0,0.730769,0.746154


### Knowledge Graph Recommendation Result

The knowledge graph score is used to rank onboarding agents according to their relationship with the selected task.

Agents with relevant skills, the same location and previous experience with the same onboarding task type receive stronger knowledge graph scores.

In [ ]:
top_5_kg = (
    kg_results[
        [
            "agent_id",
            "county",
            "skills",
            "availability_status",
            "current_workload",
            "performance_score",
            "skill_score",
            "location_score",
            "task_type_score",
            "kg_score"
        ]
    ]
    .sort_values(
        "kg_score",
        ascending=False
    )
    .head(5)
    .copy()
)

In [ ]:
top_5_kg.insert(
    0,
    "rank",
    range(1, 6)
)

top_5_kg[
    "kg_percentage"
] = (
    top_5_kg["kg_score"]
    * 100
).round(2)

top_5_kg

,rank,agent_id,county,skills,availability_status,current_workload,performance_score,skill_score,location_score,task_type_score,kg_score,kg_percentage
461,1,AGT-0462,Kakamega,"Customer Onboarding, KYC Verification, Identit...",Available,2,0.485,1.0,1.0,0.593407,0.918681,91.87
375,2,AGT-0376,Kakamega,"Customer Onboarding, KYC Verification, Identit...",Available,1,0.502,1.0,1.0,0.593407,0.918681,91.87
391,3,AGT-0392,Kakamega,"Customer Onboarding, KYC Verification, Identit...",Available,2,0.499,1.0,1.0,0.593407,0.918681,91.87
362,4,AGT-0363,Kakamega,"Customer Onboarding, KYC Verification, Face Ve...",Available,1,0.460,1.0,1.0,0.593407,0.918681,91.87
309,5,AGT-0310,Kakamega,"Customer Onboarding, KYC Verification, Identit...",Busy,5,0.591,1.0,1.0,0.593407,0.918681,91.87


# Part D: Operational Factors

## 23. Availability, Workload and Performance

A suitable agent should not only have the correct skills and previous experience.

The system should also consider whether the agent is currently available, how much work they already have and their historical performance.

Three operational factors are therefore considered:

- Availability
- Current workload
- Historical performance

These factors help reduce uneven workload distribution and avoid recommending agents who are already busy.

In [ ]:
operational_results = (
    agents.copy()
)

operational_results[
    "availability_score"
] = (
    operational_results[
        "availability_status"
    ]
    .astype(str)
    .str.lower()
    .eq("available")
    .astype(float)
)

### Workload Score

Agents with a lower current workload receive a higher workload score.

This encourages the recommendation system to distribute tasks more fairly instead of continuously selecting heavily loaded agents.

In [ ]:
max_workload = (
    operational_results[
        "current_workload"
    ].max()
)

if max_workload == 0:
    max_workload = 1

In [ ]:
operational_results[
    "workload_score"
] = (
    1
    -
    (
        operational_results[
            "current_workload"
        ]
        / max_workload
    )
)

In [ ]:
operational_results[
    "performance_score"
] = (
    operational_results[
        "performance_score"
    ]
    .fillna(
        operational_results[
            "performance_score"
        ].mean()
    )
    .clip(0, 1)
)

In [ ]:
operational_results[
    "operational_score"
] = (
    0.40
    * operational_results[
        "availability_score"
    ]

    + 0.35
    * operational_results[
        "workload_score"
    ]

    + 0.25
    * operational_results[
        "performance_score"
    ]
)

In [ ]:
operational_results[
    [
        "agent_id",
        "availability_status",
        "availability_score",
        "current_workload",
        "workload_score",
        "performance_score",
        "operational_score"
    ]
].head(10)

,agent_id,availability_status,availability_score,current_workload,workload_score,performance_score,operational_score
0,AGT-0001,Available,1.0,0,1.0,0.489,0.87225
1,AGT-0002,Busy,0.0,5,0.0,0.512,0.12800
2,AGT-0003,Available,1.0,3,0.4,0.512,0.66800
3,AGT-0004,Available,1.0,3,0.4,0.498,0.66450
4,AGT-0005,Available,1.0,2,0.6,0.576,0.75400
5,AGT-0006,Available,1.0,1,0.8,0.471,0.79775
6,AGT-0007,Busy,0.0,5,0.0,0.496,0.12400
7,AGT-0008,Available,1.0,2,0.6,0.549,0.74725
8,AGT-0009,Busy,0.0,4,0.2,0.525,0.20125
9,AGT-0010,Available,1.0,0,1.0,0.555,0.88875


### Operational Score Result

The operational score combines availability, workload and historical performance.

Available agents with lower workloads receive stronger operational scores.

This component helps the recommender balance suitability with the practical state of each onboarding agent.

# Part E: Hybrid Recommendation Model

## 24. Combining the Recommendation Components

The final recommendation system combines the four recommendation components into a single hybrid score.

The initial weighting is:

- 35% Content-Based Score
- 25% Collaborative Filtering Score
- 25% Knowledge Graph Score
- 15% Operational Score

These are initial development weights.

The validation dataset will later be used to test different combinations and determine the best-performing hybrid weights rather than assuming that these initial values are optimal.

In [ ]:
INITIAL_WEIGHTS = {
    "content": 0.35,
    "collaborative": 0.25,
    "knowledge_graph": 0.25,
    "operational": 0.15
}

print(
    "Total weight:",
    sum(INITIAL_WEIGHTS.values())
)

Total weight: 1.0


In [ ]:
hybrid_results = (
    agents.copy()
)

In [ ]:
hybrid_results[
    "content_score"
] = content_scores

In [ ]:
hybrid_results[
    "collaborative_score"
] = sample_cf_scores

In [ ]:
kg_score_map = (
    kg_results
    .set_index("agent_id")[
        "kg_score"
    ]
)

hybrid_results[
    "kg_score"
] = (
    hybrid_results[
        "agent_id"
    ].map(kg_score_map)
)

In [ ]:
operational_score_map = (
    operational_results
    .set_index("agent_id")[
        "operational_score"
    ]
)

hybrid_results[
    "operational_score"
] = (
    hybrid_results[
        "agent_id"
    ].map(
        operational_score_map
    )
)

In [ ]:
print(
    hybrid_results[
        [
            "content_score",
            "collaborative_score",
            "kg_score",
            "operational_score"
        ]
    ].isnull().sum()
)

content_score          0
collaborative_score    0
kg_score               0
operational_score      0
dtype: int64


In [ ]:
hybrid_results[
    "hybrid_score"
] = (
    INITIAL_WEIGHTS[
        "content"
    ]
    * hybrid_results[
        "content_score"
    ]

    + INITIAL_WEIGHTS[
        "collaborative"
    ]
    * hybrid_results[
        "collaborative_score"
    ]

    + INITIAL_WEIGHTS[
        "knowledge_graph"
    ]
    * hybrid_results[
        "kg_score"
    ]

    + INITIAL_WEIGHTS[
        "operational"
    ]
    * hybrid_results[
        "operational_score"
    ]
)

In [ ]:
hybrid_results = (
    hybrid_results
    .sort_values(
        "hybrid_score",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
top_5_hybrid = (
    hybrid_results[
        [
            "agent_id",
            "county",
            "skills",
            "availability_status",
            "current_workload",
            "performance_score",
            "content_score",
            "collaborative_score",
            "kg_score",
            "operational_score",
            "hybrid_score"
        ]
    ]
    .head(5)
    .copy()
)

In [ ]:
top_5_hybrid.insert(
    0,
    "rank",
    range(1, 6)
)

In [ ]:
top_5_hybrid[
    "match_percentage"
] = (
    top_5_hybrid[
        "hybrid_score"
    ]
    * 100
).round(2)

top_5_hybrid

,rank,agent_id,county,skills,availability_status,current_workload,performance_score,content_score,collaborative_score,kg_score,operational_score,hybrid_score,match_percentage
0,1,AGT-0155,Kakamega,"Customer Onboarding, KYC Verification",Available,0,0.562,1.0,1.000000,0.917582,0.89050,0.962971,96.30
1,2,AGT-0089,Kakamega,"Customer Onboarding, KYC Verification",Available,1,0.525,1.0,0.999912,0.918681,0.81125,0.951336,95.13
2,3,AGT-0341,Kakamega,"Customer Onboarding, KYC Verification",Available,1,0.480,1.0,0.999912,0.918681,0.80000,0.949648,94.96
3,4,AGT-0166,Kisumu,"Customer Onboarding, KYC Verification",Available,0,0.567,1.0,0.995013,0.792308,0.89175,0.930593,93.06
4,5,AGT-0386,Uasin Gishu,"Customer Onboarding, KYC Verification",Available,0,0.516,1.0,0.994788,0.794505,0.87900,0.929173,92.92


## 25. Generating Recommendation Explanations

The system should not only recommend an agent but should also explain why the agent was selected.

Explanations are generated using visible relationships and operational factors such as:

- matching required skills
- same task location
- previous experience with the same task type
- current availability
- workload
- historical performance

This supports transparency in the task allocation decision.

In [ ]:
def generate_explanation(
    task,
    agent
):

    reasons = []

    required_skills = set(
        split_skills(
            task["required_skills"]
        )
    )

    agent_skills = set(
        split_skills(
            agent["skills"]
        )
    )

    matched_skills = sorted(
        required_skills.intersection(
            agent_skills
        )
    )

    if matched_skills:

        reasons.append(
            "matches required skills: "
            + ", ".join(
                matched_skills
            )
        )

    if (
        str(agent["county"])
        == str(task["county"])
    ):

        reasons.append(
            "is located in the same county"
        )

    history = historical_task_types[
        (
            historical_task_types[
                "agent_id"
            ] == agent["agent_id"]
        )
        &
        (
            historical_task_types[
                "task_type"
            ] == task["task_type"]
        )
    ]

    if len(history) > 0:

        count = int(
            history.iloc[0]["count"]
        )

        reasons.append(
            "has historical experience with "
            + str(task["task_type"])
            + f" ({count} previous tasks)"
        )

    if (
        str(
            agent[
                "availability_status"
            ]
        ).lower()
        == "available"
    ):

        reasons.append(
            "is currently available"
        )

    if (
        agent[
            "current_workload"
        ] <= 2
    ):

        reasons.append(
            "has a low current workload"
        )

    if (
        agent[
            "performance_score"
        ]
        >= agents[
            "performance_score"
        ].median()
    ):

        reasons.append(
            "has good historical performance"
        )

    return "; ".join(reasons)

In [ ]:
top_5_hybrid[
    "explanation"
] = (
    top_5_hybrid
    .apply(
        lambda row:
        generate_explanation(
            sample_task,
            row
        ),
        axis=1
    )
)

In [ ]:
final_recommendations = (
    top_5_hybrid[
        [
            "rank",
            "agent_id",
            "county",
            "availability_status",
            "current_workload",
            "match_percentage",
            "explanation"
        ]
    ]
)

final_recommendations

,rank,agent_id,county,availability_status,current_workload,match_percentage,explanation
0,1,AGT-0155,Kakamega,Available,0,96.30,"matches required skills: Customer Onboarding, ..."
1,2,AGT-0089,Kakamega,Available,1,95.13,"matches required skills: Customer Onboarding, ..."
2,3,AGT-0341,Kakamega,Available,1,94.96,"matches required skills: Customer Onboarding, ..."
3,4,AGT-0166,Kisumu,Available,0,93.06,"matches required skills: Customer Onboarding, ..."
4,5,AGT-0386,Uasin Gishu,Available,0,92.92,"matches required skills: Customer Onboarding, ..."


### Hybrid Recommendation Result

The hybrid recommendation model successfully combined content-based filtering, collaborative filtering, knowledge graph relationships and operational factors.

Each onboarding agent received a final hybrid score.

The agents with the highest hybrid scores were returned as the Top-5 recommendations.

Unlike the individual recommendation components, the hybrid model considers several factors at the same time, including skills, historical task allocation patterns, knowledge graph relationships, availability, workload and historical performance.

The system also produces a human-readable explanation showing why each agent was recommended.

The current hybrid weights are initial development weights. The validation dataset will be used in the next notebook to determine the best-performing weight combination.

## 26. Saving the Recommender Components

The developed recommendation components are saved to Google Drive.

These saved components will be used during model evaluation and later integrated into the Flask web application.

In [ ]:
recommender_components = {
    "tfidf_vectorizer":
        tfidf_vectorizer,

    "agent_skill_vectors":
        agent_skill_vectors,

    "agents":
        agents,

    "interaction_matrix":
        interaction_matrix,

    "agent_similarity":
        agent_similarity,

    "collaborative_scores":
        collaborative_scores,

    "profile_to_column":
        profile_to_column,

    "historical_task_types":
        historical_task_types,

    "knowledge_graph":
        G,

    "initial_weights":
        INITIAL_WEIGHTS
}

In [ ]:
model_file = os.path.join(
    MODEL_PATH,
    "recommender_components.joblib"
)

joblib.dump(
    recommender_components,
    model_file
)

print(
    "Saved:",
    model_file
)

Saved: /content/drive/MyDrive/task-allocation-recommender/models/recommender_components.joblib


In [ ]:
final_recommendations.to_csv(
    os.path.join(
        MODEL_PATH,
        "sample_hybrid_recommendations.csv"
    ),
    index=False
)

# Notebook 02 Summary

In this notebook, the hybrid recommendation system was developed.

The completed components are:

1. Content-based filtering using TF-IDF and cosine similarity
2. Collaborative filtering using historical task-agent assignments
3. Knowledge graph reasoning using agent, skill, location and task-type relationships
4. Operational scoring using availability, workload and historical performance
5. Hybrid recommendation scoring
6. Explainable Top-N agent recommendations

The next notebook will evaluate the recommendation system using the validation and testing datasets.

In [ ]:
print(
    "NOTEBOOK 02 COMPLETE"
)

print(
    "\nAgents evaluated:",
    len(agents)
)

print(
    "Knowledge graph nodes:",
    G.number_of_nodes()
)

print(
    "Knowledge graph edges:",
    G.number_of_edges()
)

print(
    "\nTop recommended agent:",
    final_recommendations.iloc[
        0
    ]["agent_id"]
)

print(
    "Recommendation score:",
    final_recommendations.iloc[
        0
    ]["match_percentage"],
    "%"
)

print(
    "\nSaved model:",
    model_file
)

NOTEBOOK 02 COMPLETE

Agents evaluated: 500
Knowledge graph nodes: 520
Knowledge graph edges: 2893

Top recommended agent: AGT-0155
Recommendation score: 96.3 %

Saved model: /content/drive/MyDrive/task-allocation-recommender/models/recommender_components.joblib
